# 1. Imports and Configuration

In [1]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Download necessary NLTK resources
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

# Domain-specific stopwords
custom_stopwords = {
    'allow', 'class', 'available', 'part', 'case', 'lead', 'shall', 'product', 'operate',
    'operational', 'result', 'input', 'dependent', 'preference', 'item', 'without', 'let',
    'returned', 'message', 'every', 'system', 'run', 'fully', 'major', 'reasonable',
    'software', 'user', 'able', 'ability', 'support', 'year', 'expected', 'must',
    'information', 'data', 'use', 'using', 'provide', 'successfully', 'one', 'waiter',
    'include', 'accommodate', 'event', 'technique', 'recent', 'administrator', 'search',
    'add', 'allows', 'achieve', 'way', 'outside', 'release', 'launch', 'allowed', 'entered',
    'within', 'first', 'new', 'izogn', 'wcs', 'course', 'time', 'help', 'learn', 'ccr', 'cma'
}
stop_words.update(custom_stopwords)

# 2. Load Dataset

In [2]:
# Load dataset
df_real = pd.read_csv("../datasets/PROMISE_exp.csv")
df = df_real[['RequirementText', '_class_']].copy().reset_index(drop=True)

# Filter to LDA categories
valid_categories = ['F', 'US', 'SE', 'PO', 'PE', 'SC', 'FT']
df = df[df['_class_'].isin(valid_categories)]
if df.empty:
    raise ValueError("No data remains after filtering for valid categories. Check '_class_' values.")

# Remove initial stopwords
df['RequirementText'] = df['RequirementText'].apply(
    lambda x: ' '.join([word for word in x.split() if word not in stop_words])
)
df.head()

,RequirementText,_class_
0,The refresh display 60 seconds.,PE
2,If projected readable. On 10x10 projection scr...,US
4,If projected understandable. On 10x10 projecti...,US
5,The ensure accessed authorized users. The dist...,SE
6,The intuitive self-explanatory. 90% users star...,US


# 3. Text Cleaning

In [3]:
# Text cleaning function
def clean_text(text):
    text = text.lower()
    text = re.sub(r'\W+', ' ', text)  # Remove special characters
    text = re.sub(r'\d+', '', text)   # Remove numbers
    text = re.sub(r'\b\w{1,2}\b', '', text)  # Remove short words
    text = re.sub(r'\b(cloud-based)\b', 'cloud_based', text)  # Handle specific phrases
    words = text.split()
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    return ' '.join(words)

# Apply text cleaning
df['cleaned_text'] = df['RequirementText'].apply(clean_text)
df.head()

,RequirementText,_class_,cleaned_text
0,The refresh display 60 seconds.,PE,refresh display second
2,If projected readable. On 10x10 projection scr...,US,projected readable projection screen viewer re...
4,If projected understandable. On 10x10 projecti...,US,projected understandable projection screen vie...
5,The ensure accessed authorized users. The dist...,SE,ensure accessed authorized user distinguish au...
6,The intuitive self-explanatory. 90% users star...,US,intuitive self explanatory user start display ...


# 4. Data Storage and Aggregation

In [4]:
# Save cleaned data
output_filename = "../datasets/PROMISE_exp_cleaned.csv"
df.to_csv(output_filename, index=False)
print(f"Cleaned data saved to {output_filename}")

Cleaned data saved to ../datasets/PROMISE_exp_cleaned.csv
